# 07 — Pipeline de preprocesamiento de clasificación (BD_IPSA_1940)

Implementa **exactamente** las decisiones aprobadas en los notebooks 04–06 (imputación, outliers, derivadas y
formas funcionales de clasificación) en `src/preprocesamiento_clasificacion.py`:

1. `ReglasNegocioClasificacion`: reglas fijas por fila, sin estado. Deriva `mes_cosecha` y `anio`, recodifica "sin
   dato" como NaN (`lluvias` = 0, `dosismad` = 0 o > 3, `semsmad` < 0) y recorta las lluvias a 300 mm/mes × (edad − 10).
2. `ColumnTransformer`, ajustado **solo con train**: mediana de lluvia por `grupo_tenencia` × `mes_cosecha` (respaldo
   `mes_cosecha`), indicador de lluvia faltante, medianas, `log1p`, spline de `semsmad`, escalado y OneHot.

Variantes (`construir_preprocesador_clf`): `logistica` (todas las predictoras) y `knn` (conjunto reducido por
target), más las sensibilidades de sacarosa `edad_spline` y `tasa_lluvia`.

> **Alcance:** solo CC01-1940 madurada con BONUS 250 EC y cosechada a máquina.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, make_scorer
from sklearn.model_selection import cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
from src.datos import (CV_CLF, PREDICTORAS_CLASIFICACION, TARGETS_CLASIFICACION, cargar_ipsa, grupos_hacienda_ipsa,
                       preparar_xy_clf, split_clasificacion)
from src.preprocesamiento_clasificacion import (EXCLUIDAS_KNN, FLAG_LLUVIA, ImputadorMedianaGrupo,
                                                ReglasNegocioClasificacion, construir_preprocesador_clf)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. Split por hacienda (el mismo de los notebooks 06 y siguientes)

In [2]:
ipsa = cargar_ipsa()
train, test = split_clasificacion(ipsa)
compartidas = set(grupos_hacienda_ipsa(train)) & set(grupos_hacienda_ipsa(test))
print(f"Train: {len(train):,} | Test: {len(test):,} | haciendas compartidas: {len(compartidas)}")
assert (len(train), len(test)) == (1749, 438)
assert not compartidas

Train: 1,749 | Test: 438 | haciendas compartidas: 0


## 2. `ReglasNegocioClasificacion`: conteo de cada regla

Las reglas son fijas (no aprenden nada), así que se pueden contar sobre el dataset completo y comparar con el EDA
(notebooks 04 y 05).

In [3]:
X_todo = ipsa[PREDICTORAS_CLASIFICACION]
reglas = ReglasNegocioClasificacion().fit(X_todo)
R = reglas.transform(X_todo)

conteos = pd.Series({
    "dosismad = 0 → NaN (error de captura)": int(((X_todo["dosismad"] == 0) & R["dosismad"].isna()).sum()),
    "dosismad > 3 → NaN (error de captura)": int(((X_todo["dosismad"] > 3) & R["dosismad"].isna()).sum()),
    "semsmad < 0 → NaN (error de fecha)": int(((X_todo["semsmad"] < 0) & R["semsmad"].isna()).sum()),
    "lluvias = 0 → NaN (sin dato)": int(((X_todo["lluvias"] == 0) & R["lluvias"].isna()).sum()),
    "lluvias recortadas a 300 mm/mes × (edad − 10)": int((R["lluvias"] < X_todo["lluvias"].where(X_todo["lluvias"] > 0)).sum()),
})
display(conteos.to_frame("filas"))
assert conteos.tolist() == [1, 1, 2, 685, 0], "los conteos no coinciden con el EDA (04 §5, 05 §4 y §6)"

print("Columnas de salida:", list(R.columns))
assert {"mes", "periodo"}.isdisjoint(R.columns) and {"mes_cosecha", "anio"} <= set(R.columns)
assert (R["anio"] == ipsa["periodo"] // 100).all() and (R["mes_cosecha"] == ipsa["mes"]).all()

,filas
dosismad = 0 → NaN (error de captura),1
dosismad > 3 → NaN (error de captura),1
semsmad < 0 → NaN (error de fecha),2
lluvias = 0 → NaN (sin dato),685
lluvias recortadas a 300 mm/mes × (edad − 10),0


Columnas de salida: ['edad', 'cortes', 'semsmad', 'dosismad', 'lluvias', 'grupo_tenencia', 'anio', 'mes_cosecha']


In [4]:
# Sensibilidad tasa_lluvia: mm/mes de la ventana inferida (del mes 10 al corte); con ventana < 1 mes → NaN
R_tasa = ReglasNegocioClasificacion(tasa_lluvia=True).fit(X_todo).transform(X_todo)
ventana = ipsa["edad"] - 10
print(f"tasa mm/mes: mediana {R_tasa['lluvias'].median():.0f}, máx. {R_tasa['lluvias'].max():.0f} | NaN: {R_tasa['lluvias'].isna().sum()} "
      f"(685 sin dato + {int(((ventana < 1) & (ipsa['lluvias'] > 0)).sum())} con ventana < 1 mes)")
assert R_tasa["lluvias"].max() <= 300

tasa mm/mes: mediana 52, máx. 258 | NaN: 688 (685 sin dato + 3 con ventana < 1 mes)


## 3. Preprocesador: `fit` con train, `transform` con test

In [5]:
VARIANTES = {
    "logística": dict(tipo="logistica"),
    "logística + sens. sacarosa (edad spline, tasa lluvia)": dict(tipo="logistica", edad_spline=True, tasa_lluvia=True),
    "KNN TCH": dict(tipo="knn", target="TCH"),
    "KNN sacarosa": dict(tipo="knn", target="sacarosa"),
}
X_tr, X_te = train[PREDICTORAS_CLASIFICACION], test[PREDICTORAS_CLASIFICACION]
resumen, columnas = [], {}
for nombre, kw in VARIANTES.items():
    pre = construir_preprocesador_clf(**kw).fit(X_tr)
    Z_tr, Z_te = pre.transform(X_tr), pre.transform(X_te)
    columnas[nombre] = list(Z_tr.columns)
    resumen.append({"variante": nombre, "columnas": Z_tr.shape[1], "NaN train": int(Z_tr.isna().sum().sum()),
                    "NaN test": int(Z_te.isna().sum().sum())})
    assert Z_tr.isna().sum().sum() == 0 and Z_te.isna().sum().sum() == 0
display(pd.DataFrame(resumen).set_index("variante"))
for nombre, cols in columnas.items():
    print(f"{nombre}: {cols}")

,columnas,NaN train,NaN test
variante,,,
logística,26,0,0
"logística + sens. sacarosa (edad spline, tasa lluvia)",31,0,0
KNN TCH,8,0,0
KNN sacarosa,17,0,0


logística: ['lluvias', 'edad', 'cortes', 'anio', 'grupo_tenencia_2', 'grupo_tenencia_3', 'mes_cosecha_2', 'mes_cosecha_3', 'mes_cosecha_4', 'mes_cosecha_5', 'mes_cosecha_6', 'mes_cosecha_7', 'mes_cosecha_8', 'mes_cosecha_9', 'mes_cosecha_10', 'mes_cosecha_11', 'mes_cosecha_12', 'falta_dato_lluvia', 'dosismad', 'semsmad_sp_0', 'semsmad_sp_1', 'semsmad_sp_2', 'semsmad_sp_3', 'semsmad_sp_4', 'semsmad_sp_5', 'semsmad_sp_6']
logística + sens. sacarosa (edad spline, tasa lluvia): ['lluvias', 'edad_sp_0', 'edad_sp_1', 'edad_sp_2', 'edad_sp_3', 'edad_sp_4', 'edad_sp_5', 'cortes', 'anio', 'grupo_tenencia_2', 'grupo_tenencia_3', 'mes_cosecha_2', 'mes_cosecha_3', 'mes_cosecha_4', 'mes_cosecha_5', 'mes_cosecha_6', 'mes_cosecha_7', 'mes_cosecha_8', 'mes_cosecha_9', 'mes_cosecha_10', 'mes_cosecha_11', 'mes_cosecha_12', 'falta_dato_lluvia', 'dosismad', 'semsmad_sp_0', 'semsmad_sp_1', 'semsmad_sp_2', 'semsmad_sp_3', 'semsmad_sp_4', 'semsmad_sp_5', 'semsmad_sp_6']
KNN TCH: ['lluvias', 'edad', 'cortes',

In [6]:
# El conjunto de KNN excluye exactamente las candidatas aprobadas, y la logística no excluye nada
def variables_origen(cols):
    base = {"lluvias", FLAG_LLUVIA, "dosismad", "edad", "cortes", "anio", "semsmad", "grupo_tenencia", "mes_cosecha"}
    return {b for b in base if any(c == b or c.startswith(b + "_") for c in cols)}

todas = variables_origen(columnas["logística"])
assert todas == {"lluvias", FLAG_LLUVIA, "dosismad", "edad", "cortes", "anio", "semsmad", "grupo_tenencia", "mes_cosecha"}
for t in TARGETS_CLASIFICACION:
    faltan = todas - variables_origen(columnas[f"KNN {t}"])
    print(f"KNN {t}: excluye {sorted(faltan)}")
    assert faltan == EXCLUIDAS_KNN[t]
assert all(FLAG_LLUVIA in cols for nombre, cols in columnas.items() if nombre != "KNN sacarosa")
# El indicador vale 1 exactamente donde falta el dato (lluvias = 0 en el crudo)
Z_te = construir_preprocesador_clf("logistica").fit(X_tr).transform(X_te)
assert (Z_te[FLAG_LLUVIA] == (test["lluvias"] == 0).astype(int).to_numpy()).all()
print(f"{FLAG_LLUVIA} = 1 en {Z_te[FLAG_LLUVIA].sum()} filas de test (lluvias = 0: {(test['lluvias'] == 0).sum()})")

KNN TCH: excluye ['dosismad', 'mes_cosecha', 'semsmad']
KNN sacarosa: excluye ['cortes', 'falta_dato_lluvia', 'grupo_tenencia']
falta_dato_lluvia = 1 en 136 filas de test (lluvias = 0: 136)


### 3.1 Escalado de KNN (decisión explícita)

KNN usa distancias, así que la escala de cada columna decide su peso. Se comparan cuatro configuraciones con la
misma CV de train (`CV_CLF`, k = 15, solo para diagnosticar): (a) numéricas sin escalar; (b) numéricas con
`StandardScaler` ajustado en train, sin `log1p`; (c) b + `log1p(lluvias)`, la configuración **elegida**; y (d) c +
dummies estandarizadas.

In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

def knn_variante(t, escalar_num=True, log=True, escalar_dummies=False):
    pre = construir_preprocesador_clf("knn", target=t)
    ct = pre.named_steps["columnas"]
    nuevos = []
    for nombre, trf, cols in ct.transformers:
        if nombre == "lluvias" and not log:
            trf = make_pipeline(ImputadorMedianaGrupo(["lluvias"], niveles=(("grupo_tenencia", "mes_cosecha"), ("mes_cosecha",))),
                                StandardScaler())
        if not escalar_num:
            if isinstance(trf, StandardScaler):
                trf = "passthrough"
            elif hasattr(trf, "steps"):
                trf = make_pipeline(*[p for _, p in trf.steps if not isinstance(p, StandardScaler)] or ["passthrough"])
        if escalar_dummies and nombre in ("categoricas", "falta_dato_lluvia"):
            trf = make_pipeline(trf, StandardScaler())
        nuevos.append((nombre, trf, cols))
    ct.transformers = nuevos
    return Pipeline([("pre", pre), ("clf", KNeighborsClassifier(n_neighbors=15))])

CONFIGS = {"(a) numéricas sin escalar": dict(escalar_num=False, log=False),
           "(b) numéricas escaladas, sin log1p": dict(log=False),
           "(c) b + log1p(lluvias) [elegida]": dict(),
           "(d) c + dummies escaladas": dict(escalar_dummies=True)}
filas = []
for t in TARGETS_CLASIFICACION:
    X, y, g = preparar_xy_clf(train, t)
    for nombre, kw in CONFIGS.items():
        r = cross_validate(knn_variante(t, **kw), X, y, groups=g, cv=CV_CLF,
                           scoring={"F1-macro": "f1_macro", "Kappa": make_scorer(cohen_kappa_score)}, error_score="raise")
        filas.append({"target": t, "configuración": nombre, "F1-macro": r["test_F1-macro"].mean(),
                      "Kappa": r["test_Kappa"].mean(), "Kappa sd": r["test_Kappa"].std()})
escalado = pd.DataFrame(filas).set_index(["target", "configuración"]).round(3)
display(escalado)
for t in TARGETS_CLASIFICACION:
    assert escalado.loc[(t, "(c) b + log1p(lluvias) [elegida]"), "Kappa"] > escalado.loc[(t, "(a) numéricas sin escalar"), "Kappa"]

F1-macro  Kappa  Kappa sd
target   configuración                                                
TCH      (a) numéricas sin escalar              0.325  0.025     0.047
         (b) numéricas escaladas, sin log1p     0.420  0.152     0.034
         (c) b + log1p(lluvias) [elegida]       0.427  0.155     0.030
         (d) c + dummies escaladas              0.426  0.157     0.038
sacarosa (a) numéricas sin escalar              0.442  0.169     0.027
         (b) numéricas escaladas, sin log1p     0.481  0.234     0.048
         (c) b + log1p(lluvias) [elegida]       0.502  0.270     0.038
         (d) c + dummies escaladas              0.500  0.260     0.036

**Conclusión 3.1.** **Escalar las numéricas es lo que importa:** sin escalar, `anio` (~2018) y `lluvias` (cientos de
mm) dominan la distancia y el Kappa de TCH cae casi a 0 (0.025 frente a 0.12–0.14). Escalar las dummies no mejora nada (TCH igual; sacarosa
algo peor), así que el riesgo de que los 12 meses dominaran la distancia **no se confirma**: las dummies quedan sin
escalar, por convención y con evidencia. `log1p(lluvias)` mueve el resultado dentro del ruido (sacarosa +0.02; TCH −0.01), y se aplica
porque lo pide la decisión aprobada de outliers.

## 4. Anti-leakage: lo que aprende el preprocesador sale solo de train

In [8]:
pre = construir_preprocesador_clf("logistica").fit(X_tr)
ct = pre.named_steps["columnas"]
imp = ct.named_transformers_["lluvias"].steps[0][1]
R_tr = pre.named_steps["reglas"].transform(X_tr)
R_all = pre.named_steps["reglas"].transform(X_todo)

esperado_tr = R_tr.groupby(["grupo_tenencia", "mes_cosecha"])["lluvias"].median()
esperado_all = R_all.groupby(["grupo_tenencia", "mes_cosecha"])["lluvias"].median()
aprendido = imp.medianas_[0]["lluvias"]
assert np.allclose(aprendido.reindex(esperado_tr.index), esperado_tr, equal_nan=True)
print(f"Medianas grupo × mes: coinciden con las de train; difieren de las del dataset completo en "
      f"{int((~np.isclose(aprendido.reindex(esperado_all.index), esperado_all, equal_nan=True)).sum())} de {len(esperado_all)} celdas")

med_dosis = ct.named_transformers_["dosis"].steps[0][1].statistics_[0]
assert med_dosis == R_tr["dosismad"].median()
media_edad = ct.named_transformers_["edad"].mean_[0]
assert np.isclose(media_edad, X_tr["edad"].mean())
print(f"Mediana de dosis (train) {med_dosis} | media de edad del escalador (train) {media_edad:.3f} frente a "
      f"{X_todo['edad'].mean():.3f} en el dataset completo")

# Transformar test no reajusta nada
antes = aprendido.copy()
pre.transform(X_te)
assert imp.medianas_[0]["lluvias"].equals(antes)

Medianas grupo × mes: coinciden con las de train; difieren de las del dataset completo en 30 de 36 celdas
Mediana de dosis (train) 1.0 | media de edad del escalador (train) 12.761 frente a 12.766 en el dataset completo


## 5. Smoke test: el Pipeline completo corre dentro de la CV por lote

Solo verifica que el preprocesador se reajusta en cada fold sin errores. **No son resultados de modelado:**
hiperparámetros por defecto, sin ajustar `C`, la penalización ni `k`.

In [9]:
metricas = {"F1-macro": "f1_macro", "Kappa": make_scorer(cohen_kappa_score)}
filas = []
for t in TARGETS_CLASIFICACION:
    X, y, g = preparar_xy_clf(train, t)
    modelos = {
        "logística (L2, C=1)": Pipeline([("pre", construir_preprocesador_clf("logistica")),
                                         ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))]),
        "KNN (k=15)": Pipeline([("pre", construir_preprocesador_clf("knn", target=t)),
                                ("clf", KNeighborsClassifier(n_neighbors=15))]),
    }
    for nombre, modelo in modelos.items():
        with warnings.catch_warnings():
            warnings.simplefilter("error", category=UserWarning)  # p. ej. categorías desconocidas en un fold
            r = cross_validate(modelo, X, y, groups=g, cv=CV_CLF, scoring=metricas, error_score="raise")
        filas.append({"target": t, "modelo": nombre, "F1-macro": r["test_F1-macro"].mean(), "Kappa": r["test_Kappa"].mean()})
display(pd.DataFrame(filas).set_index(["target", "modelo"]).round(3))

F1-macro  Kappa
target   modelo                              
TCH      logística (L2, C=1)     0.427  0.155
         KNN (k=15)              0.427  0.155
sacarosa logística (L2, C=1)     0.458  0.225
         KNN (k=15)              0.502  0.270

## Conclusión

- Las reglas fijas reproducen los conteos del EDA (1 + 1 dosis, 2 semanas, 685 lluvias sin dato, 0 recortes).
- `mes`/`periodo` se reemplazan por `mes_cosecha` y `anio`. Esto formaliza la derivación ad hoc de los notebooks
  04–06.
- El preprocesador aprende medianas, medias y categorías solo de train. Transformar test no reajusta nada, y dentro
  de la CV se reajusta en cada fold.
- KNN excluye exactamente las candidatas aprobadas con el split por hacienda (TCH: `semsmad`, `mes_cosecha`,
  `dosismad`; sacarosa: `cortes`, `grupo_tenencia`, `falta_dato_lluvia`).
- Escalado de KNN decidido (§3.1): numéricas con StandardScaler de train + `log1p(lluvias)`; dummies sin escalar.
- El indicador de lluvia se llama `falta_dato_lluvia` y vale 1 cuando falta el dato.